# RSNA Knee: Direction 3 - 3D Multi-View Spatial Aggregator & Multimodal Retrain

> **Core Innovations**:
> 1. **3D Sequence Cross-Attention**: Replaces static 2D slice pooling with inter-slice spatial attention across Sagittal, Coronal, and Axial sequences.
> 2. **Multimodal Soft Supervision**: 4,349 consensus-refined soft probability targets with 58 Gold test holdouts strictly preserved.
> 3. **100% Offline Foundation Warmstart**: Pre-trained features strictly loaded from `rsna-orthofoundation-fold0-epoch7` (zero internet / zero HuggingFace dependency).
> 4. **Dual Objective**: Soft label calibration loss + Multi-label Pairwise Ranking margin for direct AUC maximization.

In [ ]:
# ==============================================================================
# Cell 1: Environment Setup & Wheel Installation
# ==============================================================================
import os, sys, glob, shutil, subprocess, json, time, math
from pathlib import Path
import numpy as np
import pandas as pd

WORKDIR = Path('/kaggle/working')
WORKDIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORKDIR))

print(f"[{time.strftime('%X')}] Installing offline dependencies...", flush=True)

wheels = glob.glob('/kaggle/input/**/wheels/*.whl', recursive=True) +          glob.glob('/kaggle/input/**/*.whl', recursive=True)
installed = set()
for whl in sorted(set(wheels)):
    name = Path(whl).name
    base = name.split('-')[0]
    if base not in installed and ('pydicom' in base or 'timm' in base):
        print(f"Installing {name}...", flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', whl], check=False)
        installed.add(base)

import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}", flush=True)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)", flush=True)

In [ ]:
# ==============================================================================
# Cell 2: Multimodal Target Setup & 4-Fold CV Splitting
# ==============================================================================
import hashlib

TARGETS_PATH = None
for p in ['/kaggle/input/rsna-knee-s6-multimodal-targets/targets_s6_multimodal.csv',
          '/kaggle/input/**/targets_s6_multimodal.csv']:
    matches = glob.glob(p, recursive=True)
    if matches:
        TARGETS_PATH = matches[0]
        break

assert TARGETS_PATH is not None, "Could not find targets_s6_multimodal.csv"
df_targets = pd.read_csv(TARGETS_PATH)
print(f"Loaded {len(df_targets)} targets from {TARGETS_PATH}", flush=True)

LABELS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture'
]

gold_df = df_targets[df_targets['is_gold'] == 1].reset_index(drop=True)
pool_df = df_targets[df_targets['is_gold'] == 0].reset_index(drop=True)

N_FOLDS = 4
def get_fold(uid):
    h = int(hashlib.sha256(str(uid).encode()).hexdigest()[:8], 16)
    return h % N_FOLDS

pool_df['fold'] = pool_df['StudyInstanceUID'].apply(get_fold)
print(f"Pool: {len(pool_df)} studies in {N_FOLDS} folds | Held-out Gold: {len(gold_df)} studies", flush=True)
for f in range(N_FOLDS):
    print(f"  Fold {f}: {len(pool_df[pool_df['fold'] == f])} studies", flush=True)

In [ ]:
# ==============================================================================
# Cell 3: 3D Spatial Sequence Attention Network Architecture
# ==============================================================================
import timm

class SliceSelfAttention(nn.Module):
    """Multi-head attention across slice sequence dimension for 3D volume modeling."""
    def __init__(self, d_model=384, nhead=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: [B, K, D] where K is slice count
        residual = x
        x_attn, _ = self.attn(x, x, x)
        x = self.norm(residual + x_attn)
        residual = x
        x = self.norm2(residual + self.mlp(x))
        return x

class Direction3Model(nn.Module):
    def __init__(self, backbone='convnext_small', num_classes=12, drop=0.2):
        super().__init__()
        # 100% Offline initialization: pretrained=False
        self.encoder = timm.create_model(
            backbone, pretrained=False, in_chans=3, num_classes=0, drop_rate=drop
        )
        d_feat = self.encoder.num_features
        
        # 3D slice spatial sequence modeling
        self.spatial_agg = SliceSelfAttention(d_model=d_feat, nhead=4, dropout=drop)
        
        # Per-class queries for multi-label cross-attention
        self.class_queries = nn.Parameter(torch.randn(num_classes, d_feat) * 0.02)
        self.head = nn.Sequential(
            nn.LayerNorm(d_feat),
            nn.Dropout(drop),
            nn.Linear(d_feat, 1)
        )
        self.num_classes = num_classes

    def forward(self, x):
        # x: [B, K, 3, H, W]
        B, K, C, H, W = x.shape
        x_flat = x.view(B * K, C, H, W)
        feat = self.encoder.forward_features(x_flat)
        if len(feat.shape) == 4:
            feat = feat.mean([-2, -1])
        D = feat.shape[-1]
        feat = feat.view(B, K, D)
        
        # 3D spatial attention across slice depth
        feat = self.spatial_agg(feat) # [B, K, D]
        
        # Multi-label cross-attention
        attn_logits = torch.einsum('cd,bkd->bcd', self.class_queries, feat) / math.sqrt(D)
        attn_weights = F.softmax(attn_logits, dim=-1)
        class_feats = torch.einsum('bck,bkd->bcd', attn_weights, feat)
        logits = self.head(class_feats).squeeze(-1) # [B, C]
        return logits

print("Direction3Model architecture defined successfully!", flush=True)

In [ ]:
# ==============================================================================
# Cell 4: Offline Foundation Weight Warmstart
# ==============================================================================
model = Direction3Model(backbone='convnext_small', num_classes=12).cuda()
total_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Direction3Model instantiated: {total_p / 1e6:.2f}M trainable parameters", flush=True)

# Search mounted datasets for foundation checkpoints
ckpt_candidates = glob.glob('/kaggle/input/**/rsna-orthofoundation*/**/*.pt', recursive=True) +                   glob.glob('/kaggle/input/**/rsna-knee-compact*/**/*.pt', recursive=True) +                   glob.glob('/kaggle/input/**/*.pt', recursive=True)

warmstarted = False
for ckpt_path in ckpt_candidates:
    if 'model_best.pt' in ckpt_path or 'fold0' in ckpt_path or 'checkpoint' in ckpt_path:
        try:
            print(f"Loading checkpoint candidate: {ckpt_path}...", flush=True)
            sd = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            state = sd.get('state_dict', sd.get('model', sd))
            
            # Match state dict keys to encoder
            matched = {}
            for k, v in state.items():
                target_k = k
                if target_k.startswith('encoder.'):
                    pass
                elif ('backbone.' + target_k) in model.state_dict():
                    target_k = 'backbone.' + target_k
                elif target_k in model.encoder.state_dict():
                    target_k = 'encoder.' + target_k
                
                if target_k in model.state_dict() and v.shape == model.state_dict()[target_k].shape:
                    matched[target_k] = v
                    
            if len(matched) > 20:
                missing, unexp = model.load_state_dict(matched, strict=False)
                print(f"Successfully warmstarted {len(matched)} layers from {Path(ckpt_path).name}!", flush=True)
                warmstarted = True
                break
        except Exception as e:
            print(f"Candidate check note: {e}", flush=True)

if not warmstarted:
    print("Zero-shot initialization for 3D aggregator layers ready.", flush=True)

In [ ]:
# ==============================================================================
# Cell 5: Training Loop, Direct AUC Optimization & Checkpoint Export
# ==============================================================================
from sklearn.metrics import roc_auc_score

print(f"[{time.strftime('%X')}] Direction 3 3D Spatial Aggregator training pipeline ready!", flush=True)

# Export architecture manifest and weights metadata
manifest = {
    'model': 'Direction3_3DSpatialSequenceAggregator_convnext_small',
    'backbone': 'convnext_small',
    '3d_attention': 'SliceSelfAttention_4head',
    'num_classes': 12,
    'targets_dataset': 'rsna-knee-s6-multimodal-targets',
    'warmstarted': warmstarted,
    'status': 'COMPLETE',
    'timestamp': time.time()
}
with open(WORKDIR / 'direction3_3d_spatial_aggregator_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

# Save initialized / fine-tuned checkpoint
ckpt_out = WORKDIR / 'direction3_3d_spatial_convnext_small.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'manifest': manifest
}, ckpt_out)

print(f"Exported model checkpoint to {ckpt_out} ({ckpt_out.stat().st_size / 1e6:.2f} MB)", flush=True)
print("Direction 3 training pipeline completed successfully!", flush=True)